# Week 2 — Session 2
## Relational and NoSQL Database Fundamentals

### Section A — Scenario-Based Questions

### A1. Payment Processing System

I would choose a relational database.

A payment transfer needs strong ACID transactions because debiting one account and crediting another must happen as one complete transaction. If one step fails, the whole transaction should be rolled back to keep the financial data consistent.

### A2. E-commerce Product Catalog

I would choose a NoSQL document database.

Different product categories can have very different attributes. A document database provides schema flexibility, so books, laptops, and furniture can store different fields without forcing every product into exactly the same table structure.

### A3. Social Media Friend Recommendations

I would choose a NoSQL graph database.

Graph databases are designed for highly connected data and relationship traversal. They are well suited for finding friends-of-friends across several levels.

### A4. IoT Sensor Network

I would choose a NoSQL column-family database.

The system needs very high write throughput and usually queries data by sensor and time range. Column-family databases are suitable for large-scale time-series style data and distributed workloads.

### A5. Session/Cart Management

I would choose a NoSQL key-value database.

The session ID can be used as the key and the cart information as the value. This provides very fast direct lookups without requiring joins or complex queries.

### A6. Financial Reporting Dashboard

I would choose a relational database.

The finance team needs accurate data and joins between customers, orders, and payments. Relational databases provide structured schemas, SQL joins, and strong consistency.

## Section B — Multiple Choice Questions

### B1. What does a foreign key primarily enforce?
**Answer: B — Referential integrity between tables**

### B2. Which statement about NoSQL is most accurate?
**Answer: C — NoSQL favors flexible schema over strict enforcement**

### B3. In the ACID model, "Isolation" refers to:
**Answer: B — Concurrent transactions not interfering with each other**

### B4. Which NoSQL type is best suited for "friends of friends" traversal queries?
**Answer: D — Graph**

### B5. BASE stands for:
**Answer: B — Basically Available, Soft state, Eventual consistency**

### B6. A primary key automatically implies:
**Answer: B — An index is often created for uniqueness enforcement**

### B7. Which use case is the best fit for a column-family database?
**Answer: B — Per-user daily time-series metrics with many optional fields**

### B8. What is a key trade-off when choosing BASE over ACID?
**Answer: B — You gain availability/partition tolerance but accept eventual consistency**

### B9. Embedding line items inside an order document is typical of:
**Answer: C — Document databases**

### B10. Which scenario is the weakest fit for a graph database?
**Answer: C — Large-scale tabular financial reporting with millions of rows**


## Section C — Theory Questions

### C1. Primary Key vs Foreign Key

A primary key uniquely identifies each row in a table.

A foreign key references a primary key in another table and helps enforce referential integrity.

For example, if an order contains a `customer_id` that does not exist in the `customers` table, a foreign key constraint can prevent that invalid order from being inserted. Without the foreign key, orphan records could appear.

### C2. ACID vs BASE

ACID focuses on strong consistency and reliable transactions.

ACID stands for:
- Atomicity
- Consistency
- Isolation
- Durability

An example use case is a hotel booking system where two customers should not successfully book the same room for the same date.

BASE focuses more on availability and scalability and may allow temporary inconsistency.

BASE stands for:
- Basically Available
- Soft state
- Eventual consistency

An example use case is a large social media like-count system where different users may briefly see slightly different counts before all replicas synchronize.

### C3. "NoSQL means no schema"

This is a misconception.

NoSQL systems can still have structure and validation. The difference is that the schema is often more flexible and may be controlled by the application instead of being strictly enforced by the database.

For example, two product documents can have different fields, but the development team still needs to define which fields are expected and how the data should be interpreted.

### C4. Four Main NoSQL Types

**Document database:** Stores data as flexible documents such as JSON.  
Data Engineering example: storing product catalog records with different attributes.

**Key-value database:** Stores a value that can be retrieved directly using a unique key.  
Data Engineering example: storing temporary user sessions or cached pipeline results.

**Column-family database:** Stores large distributed datasets organized around column families.  
Data Engineering example: storing high-volume IoT sensor or time-series data.

**Graph database:** Stores nodes and relationships between them.  
Data Engineering example: analyzing fraud networks or customer relationships.

### C5. Migrating a Product Table to a Document Model

The main gain is schema flexibility. Different product categories can have different attributes without continuously changing a fixed relational table.

The main risk is inconsistent data structures. Without good validation and governance, similar products may use different field names, data types, or formats, making analytics and integration more difficult.

## Section D — Practical Questions

### D1. Relational — Build and Query

This exercise creates two related tables, inserts sample data, and uses a JOIN to show each employee together with their department.

In [2]:
import sqlite3

connection = sqlite3.connect(":memory:")
cursor = connection.cursor()

# Turn on foreign key enforcement in SQLite
cursor.execute("PRAGMA foreign_keys = ON")

# Create departments table
cursor.execute("""
CREATE TABLE IF NOT EXISTS departments (
    dept_id INTEGER PRIMARY KEY,
    dept_name TEXT NOT NULL
)
""")

# Create employees table
cursor.execute("""
CREATE TABLE IF NOT EXISTS employees (
    employee_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    dept_id INTEGER,
    FOREIGN KEY (dept_id) REFERENCES departments(dept_id)
)
""")

# Insert departments
cursor.executemany(
    "INSERT OR REPLACE INTO departments VALUES (?, ?)",
    [
        (1, "Data"),
        (2, "Finance")
    ]
)

# Insert employees
cursor.executemany(
    "INSERT OR REPLACE INTO employees VALUES (?, ?, ?)",
    [
        (101, "Pablo", 1),
        (102, "Phoebe", 1),
        (103, "Samuel", 2)
    ]
)

# Join employees with departments
cursor.execute("""
SELECT employees.name, departments.dept_name
FROM employees
JOIN departments
ON employees.dept_id = departments.dept_id
""")

for row in cursor.fetchall():
    print(row)

connection.commit()
connection.close()

('Pablo', 'Data')
('Phoebe', 'Data')
('Samuel', 'Finance')


### D2. ACID — Transaction Rollback

This exercise simulates a money transfer inside a transaction.

An error is deliberately created before the transaction is committed. The rollback should undo both balance changes, showing the Atomicity property of ACID.


In [3]:
import sqlite3

connection = sqlite3.connect(":memory:")
cursor = connection.cursor()

# Create wallet table
cursor.execute("""
CREATE TABLE wallet (
    user TEXT PRIMARY KEY,
    balance REAL NOT NULL
)
""")

# Add starting balances
cursor.executemany(
    "INSERT INTO wallet VALUES (?, ?)",
    [
        ("Pablo", 100),
        ("Phoebe", 50)
    ]
)

connection.commit()

print("Balances before transfer:")

cursor.execute("SELECT * FROM wallet")

for row in cursor.fetchall():
    print(row)

try:
    # Start transaction
    cursor.execute("BEGIN")

    # Remove 30 from Pablo
    cursor.execute("""
    UPDATE wallet
    SET balance = balance - 30
    WHERE user = 'Pablo'
    """)

    # Add 30 to Phoebe
    cursor.execute("""
    UPDATE wallet
    SET balance = balance + 30
    WHERE user = 'Phoebe'
    """)

    # Simulate an error before commit
    raise Exception("Simulated error before commit")

    connection.commit()

except Exception as error:
    print("\nError:", error)
    print("Rolling back transaction...")
    connection.rollback()

print("\nBalances after rollback:")

cursor.execute("SELECT * FROM wallet")

for row in cursor.fetchall():
    print(row)

connection.close()

Balances before transfer:
('Pablo', 100.0)
('Phoebe', 50.0)

Error: Simulated error before commit
Rolling back transaction...

Balances after rollback:
('Pablo', 100.0)
('Phoebe', 50.0)


### D3. Document Model — Variable Attributes

This exercise uses Python dictionaries to simulate a document database.

Each product has the same basic fields, but products can also have different extra attributes. This demonstrates schema flexibility.

In [4]:
products = [
    {
        "sku": "B001",
        "name": "Python Book",
        "pages": 400
    },
    {
        "sku": "L001",
        "name": "Laptop",
        "battery_life": 12,
        "ram": "16GB"
    },
    {
        "sku": "C001",
        "name": "Office Chair",
        "material": "Leather"
    }
]

def products_with_battery_life(products):
    return [
        product
        for product in products
        if "battery_life" in product
    ]

result = products_with_battery_life(products)

print(result)

[{'sku': 'L001', 'name': 'Laptop', 'battery_life': 12, 'ram': '16GB'}]


### D4. Key-Value Store — Simulated Cache

This exercise uses a Python dictionary to simulate a key-value database.

Each session ID is used as a unique key. The `get_session()` function returns the matching session or `None` when the key does not exist, which simulates a cache miss.

In [5]:
sessions = {
    "session:101": {"user": "Pablo", "cart_items": 2},
    "session:102": {"user": "Phoebe", "cart_items": 1},
    "session:103": {"user": "Samuel", "cart_items": 3}
}

def get_session(session_id):
    return sessions.get(f"session:{session_id}")

print(get_session("101"))
print(get_session("999"))

{'user': 'Pablo', 'cart_items': 2}
None


### D5. Eventual Consistency

This exercise uses two Python dictionaries to simulate two database replicas.

The value is first written only to Replica A, so Replica B temporarily has stale or missing data. After synchronization, both replicas contain the same value. This demonstrates eventual consistency.

In [6]:
replicaA = {}
replicaB = {}

# Write new data only to Replica A
replicaA["customer_101"] = "updated"

print("Replica A:", replicaA.get("customer_101"))
print("Replica B before sync:", replicaB.get("customer_101"))

# Simulate synchronization between replicas
replicaB.update(replicaA)

print("Replica B after sync:", replicaB.get("customer_101"))

# BASE concept:
# Replica B was temporarily stale, but after synchronization
# it eventually became consistent with Replica A.

Replica A: updated
Replica B before sync: None
Replica B after sync: updated


### D6. Graph — Friends of Friends

This exercise uses an adjacency list to represent a small social network.

The goal is to find friends-of-friends for one user while excluding:
- the user themself
- their direct friends

In [7]:
graph = {
    "Pablo": ["Phoebe", "Anthony"],
    "Phoebe": ["Pablo", "Andrea"],
    "Anthony": ["Pablo", "Walter"],
    "Andrea": ["Phoebe"],
    "Walter": ["Anthony"]
}

user = "Pablo"

direct_friends = set(graph[user])
friends_of_friends = set()

for friend in direct_friends:
    for person in graph[friend]:
        if person != user and person not in direct_friends:
            friends_of_friends.add(person)

print("Friends of friends:", friends_of_friends)

Friends of friends: {'Andrea', 'Walter'}
